# Weather-or-Not: Flight Delay Classifier
### Naive Bayes vs. Feedforward Neural Network

**Target:** Predict whether a flight will arrive delayed (≥15 min) based on weather at origin & destination airports.

**Pipeline:**
1. Load flight + weather data
2. Merge weather onto flights (origin & destination)
3. Feature engineering
4. Train / test split
5. Model with FFN


Imports & Config

In [9]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import holidays
from datetime import timedelta

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, make_scorer
from sklearn.utils import resample
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, TargetEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 50)

with open('params_file.json') as f:
    params = json.load(f)

FLIGHT_CSV      = params['Output_Directory'] + 'flights_dataset.csv'
IEM_WEATHER_DIR = params['Dataset_Directory'] + '2024_iem_weather'
DELAY_THRESHOLD = 15        # FAA standard: 15+ min = delayed
RANDOM_STATE    = 1354
TEST_SIZE       = 0.2

#Same loading as with NB
CACHE_PATH = 'flights_with_weather.parquet'



## 1. Load Data

In [10]:
%%time

if os.path.exists(CACHE_PATH):
    print('Loading cached merged dataset...')
    merged = pd.read_parquet(CACHE_PATH)
else:
    print('Joining weather onto flights (this may take a few minutes)...')
    weather_records = []
    for _, row in flights.iterrows():
        orig_wx = get_interpolated_weather(row.get('ORIGIN',''), row['DEP_TS'], iem_weather, 'ORIG')
        dest_wx = get_interpolated_weather(row.get('DEST',''),   row['ARR_TS'], iem_weather, 'DEST')
        weather_records.append({**orig_wx, **dest_wx})
    wx_df  = pd.DataFrame(weather_records, index=flights.index)
    merged = pd.concat([flights, wx_df], axis=1)
    merged.to_parquet(CACHE_PATH, index=False)
    print(f'Cached to {CACHE_PATH}')

print(f'Merged shape: {merged.shape}')

Loading cached merged dataset...
Merged shape: (6510337, 58)
CPU times: total: 6.75 s
Wall time: 1.8 s


## 4. Feature Engineering

In [ ]:
# Temporal features
WEATHER_COLS = ['tmpf', 'dwpf', 'relh', 'drct', 'sknt', 'vsby', 'mslp', 'gust']

merged['DEP_HOUR']   = merged['DEP_TS'].dt.hour
merged['DEP_DOW']    = merged['DEP_TS'].dt.dayofweek
merged['DEP_MONTH']  = merged['DEP_TS'].dt.month
merged['IS_WEEKEND'] = (merged['DEP_DOW'] >= 5).astype(int)
merged['RUSH_HOUR']  = merged['DEP_HOUR'].apply(
    lambda h: 1 if h in range(7, 10) or h in range(16, 20) else 0
)

#Grab next holdays distance
def get_days_to_next_holiday(df, timestamp_col='DEP_TS'):
    print(df[timestamp_col][0])
    print(pd.to_datetime(df[timestamp_col][0]))
    df[timestamp_col] = pd.to_datetime(df[timestamp_col])
    #only 2024 for our dataset but good it be sure
    years = df[timestamp_col].dt.year.unique()
    #only US holidays
    us_holidays = holidays.US(years=list(years) + [max(years) + 1])
    holiday_dates = sorted(us_holidays.keys())
    holiday_series = pd.to_datetime(holiday_dates)

    def find_next(ts):
        future_holidays = holiday_series[holiday_series >= ts.normalize()]
        if not future_holidays.empty:
            #grab closest holiday in the future
            next_h = future_holidays[0]
            return (next_h - ts).days
        return np.nan

    return df[timestamp_col].apply(find_next)

#Flight Denisty Funciton (will be slightly difficult to get from flight aware API)
def add_flight_density(df):
    #Get the orign/
    df = df.sort_values(['ORIGIN', 'DEP_TS'])
    
    #Grab flights within a 2 hour window at that airport
    density = (
        df.set_index('DEP_TS')
        .groupby('ORIGIN')['FLIGHTS'] # 'FLIGHTS' column is usually just 1s
        .rolling('2h', center=True)
        .count()
        .reset_index(drop=True)
    )
    
    return density.values



# Feature lists
CONTINUOUS_FEATURES = [
    'ORIG_tmpf','ORIG_dwpf','ORIG_relh','ORIG_sknt','ORIG_vsby','ORIG_mslp',
    'DEST_tmpf','DEST_dwpf','DEST_relh','DEST_sknt','DEST_vsby','DEST_mslp',
    'DELTA_tmpf','DELTA_sknt','DELTA_vsby',
    'DEP_HOUR_SIN', 'DEP_HOUR_COS', 'DAYS_TO_HOLIDAY', 'FLIGHT_DENSITY','IS_WEEKEND','RUSH_HOUR',
    'ORIG_LOW_VIS','ORIG_HIGH_WIND','ORIG_GUSTING',
    'DEST_LOW_VIS','DEST_HIGH_WIND','DEST_GUSTING',
]

#Theese will need embeddings
CATEGORICAL_FEATURES = [
    'DEP_DOW','DEP_MONTH'
]
ALL_FEATURES = CONTINUOUS_FEATURES + CATEGORICAL_FEATURES
TARGET = 'DELAYED'

CACHE_PATH = 'flights_with_weather_features.parquet'

if os.path.exists(CACHE_PATH):
    print('Loading cached merged dataset...')
    merged = pd.read_parquet(CACHE_PATH)
else:
    # Apply to dataframe
    print("Days To Holiday Feature:")
    %time
    merged['DAYS_TO_HOLIDAY'] = get_days_to_next_holiday(merged, 'DEP_TS')
    print("Flight Density Feature:")
    %time
    merged['FLIGHT_DENSITY']= add_flight_density(merged)

    print("Weather Delta Features")
    # Weather delta (destination - origin)
    for col in WEATHER_COLS:
        o, d = f'ORIG_{col}', f'DEST_{col}'
        if o in merged.columns and d in merged.columns:
            merged[f'DELTA_{col}'] = merged[d] - merged[o]

    print("Weather Flag Features")
    # Derived flags
    for pfx in ['ORIG', 'DEST']:
        merged[f'{pfx}_LOW_VIS']   = (merged[f'{pfx}_vsby'] < 3).astype(float)
        merged[f'{pfx}_HIGH_WIND'] = (merged[f'{pfx}_sknt'] > 20).astype(float)
        merged[f'{pfx}_GUSTING']   = merged[f'{pfx}_gust'].notna().astype(float)
    #convert hour into sin
    merged['DEP_HOUR_SIN'] = np.sin(2 * np.pi * merged['DEP_HOUR']/24)
    merged['DEP_HOUR_COS'] = np.cos(2 * np.pi * merged['DEP_HOUR']/24)

#Null Handling
for feature in CATEGORICAL_FEATURES:
    merged[feature].fillna(-1, inplace = True)
for feature in CONTINUOUS_FEATURES:
    #Mean imputation for now
    merged[feature].fillna(merged[feature].mean(), inplace = True)

merged.to_parquet(CACHE_PATH, index=False)
print(f'Cached to {CACHE_PATH}')




Loading cached merged dataset...
Cached to flights_with_weather_features.parquet


## Output Labels

In [12]:
TARGET = 'ARR_DELAY'
  
print(merged[TARGET].unique())

merged['Labels'] = merged[TARGET] // 15

merged['Labels'].clip(lower=0, upper=1, inplace=True)

merged['Labels'].fillna(0, inplace = True)

print(merged['Labels'].unique())



[ -17.  -33.   -8. ... 1744. 1884. 2495.]
[0. 1.]


In [13]:
print(merged.columns)

Index(['Unnamed: 0', 'YEAR', 'FL_DATE', 'OP_UNIQUE_CARRIER',
       'OP_CARRIER_AIRLINE_ID', 'OP_CARRIER', 'TAIL_NUM', 'OP_CARRIER_FL_NUM',
       'ORIGIN_AIRPORT_ID', 'ORIGIN_AIRPORT_SEQ_ID', 'ORIGIN_CITY_MARKET_ID',
       'ORIGIN', 'ORIGIN_WAC', 'DEST_AIRPORT_ID', 'DEST_AIRPORT_SEQ_ID',
       'DEST_CITY_MARKET_ID', 'DEST', 'DEST_WAC', 'DEP_TIME', 'DEP_DELAY',
       'DEP_DELAY_GROUP', 'ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_GROUP',
       'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'AIR_TIME', 'FLIGHTS',
       'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY',
       'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'source_file',
       'ORIGIN_WEATHER_STATION', 'DEST_WEATHER_STATION', 'DELAYED',
       'FL_DATE_CLEAN', 'DEP_TS', 'ARR_TS', 'ORIG_tmpf', 'ORIG_dwpf',
       'ORIG_relh', 'ORIG_drct', 'ORIG_sknt', 'ORIG_vsby', 'ORIG_mslp',
       'ORIG_gust', 'DEST_tmpf', 'DEST_dwpf', 'DEST_relh', 'DEST_drct',
       'DEST_sknt', 'DEST_vsby', 'DEST_mslp', 'DEST_gust', 'DEP_HOUR',
      

In [14]:
model_df = merged.copy()
print(f'Modelling dataset: {len(model_df):,} rows | Delay rate: {model_df['Labels'].mean():.1%}')

if 'CANCELLED' in model_df.columns:
    model_df = model_df[model_df['CANCELLED'] == 0]
if 'DIVERTED' in model_df.columns:
    model_df = model_df[model_df['DIVERTED'] == 0]
model_df = model_df[ALL_FEATURES + ['Labels']].dropna(subset=['Labels'])

Modelling dataset: 6,510,337 rows | Delay rate: 20.7%


## Pipeline Setup

In [ ]:

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), CONTINUOUS_FEATURES), 
        ('cat', TargetEncoder(target_type='continuous'), CATEGORICAL_FEATURES)
    ])

#FFN Def
ffn = MLPClassifier(
    hidden_layer_sizes=(512, 256, 128),
    activation='relu', 
    solver='adam',
    max_iter=500,
    early_stopping=True,
    random_state=5,
    batch_size=512,
    validation_fraction=0.15,
    n_iter_no_change=5
)

#pipeline
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', ffn)
])

In [ ]:
#Train / Test Dataset Setup
#X
label_encoder = LabelEncoder()
X = model_df[CONTINUOUS_FEATURES + CATEGORICAL_FEATURES]
y = label_encoder.fit_transform(model_df['Labels'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=27)

# Create a balanced training set by downsampling the majority class
df_train = pd.concat([X_train, pd.Series(y_train, name='label', index=X_train.index)], axis=1)

# # Separate the massive no delay class
# df_class_0 = df_train[df_train.label == 0]
# df_others = df_train[df_train.label != 0]

# # Downsample class 0 to match the total count of all other classes combined
# # df_class_0_downsampled = resample(df_class_0, 
# #                                   replace=False, 
                   
                   
# #                                   n_samples=len(df_others), 
# #                                   random_state=27)

# df_balanced = pd.concat([df_class_0, df_others])

# Now use df_balanced to train your model
X_train_bal = df_train.drop('label', axis=1)
y_train_bal = df_train['label']

#Train the model
# This automatically runs the data through the scaler/encoder, then trains the FFN

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_bal)


#Original single-model
print("Training model...")
model_pipeline.fit(X_train_bal, y_train_bal, classifier__sample_weight=sample_weights)
print("Training complete.\n")



In [ ]:
#Grid Search for FFN Hyperparameters
param_grid = {
    'classifier__hidden_layer_sizes': [
        (128, 64),
        (256, 128, 64),
        (256, 128, 64, 32),
        (512, 256, 128),
    ],
    'classifier__activation': ['relu', 'tanh'],
    'classifier__alpha': [1e-4, 1e-3, 1e-2],
    'classifier__learning_rate_init': [1e-3, 5e-4],
}

f1_delayed = make_scorer(f1_score, pos_label=1)

grid_search = GridSearchCV(
    estimator=model_pipeline,
    param_grid=param_grid,
    scoring=f1_delayed,
    cv=3,                    
    n_jobs=-1,               
    verbose=2,
    refit=True,
    return_train_score=True,
)

print(f"Grid contains {sum(1 for _ in __import__('itertools').product(*param_grid.values()))} "
      f"parameter combinations × 3 folds = "
      f"{sum(1 for _ in __import__('itertools').product(*param_grid.values())) * 3} fits.")
print("Starting grid search — this will take a while...")
grid_search.fit(X_train_bal, y_train_bal,
                classifier__sample_weight=sample_weights)

print("\nGrid Search Results")
print(f"Best CV F1 (delayed class): {grid_search.best_score_:.4f}")
print(f"Best params:  {grid_search.best_params_}")

#results taable
cv_results = __import__('pandas').DataFrame(grid_search.cv_results_)
cols = ['params', 'mean_test_score', 'std_test_score', 'mean_train_score', 'rank_test_score']
print("\nTop-5 configurations:")
print(
    cv_results[cols]
    .sort_values('rank_test_score')
    .head(5)
    .to_string(index=False)
)


Grid contains 48 parameter combinations × 3 folds = 144 fits.
Starting grid search — this will take a while...
Fitting 3 folds for each of 48 candidates, totalling 144 fits

Grid Search Results
Best CV F1 (delayed class): 0.4605
Best params:  {'classifier__activation': 'relu', 'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (512, 256, 128), 'classifier__learning_rate_init': 0.001}

Top-5 configurations:
                                                                                                                                                      params  mean_test_score  std_test_score  mean_train_score  rank_test_score
 {'classifier__activation': 'relu', 'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (512, 256, 128), 'classifier__learning_rate_init': 0.001}         0.460545        0.001112          0.475636                1
  {'classifier__activation': 'relu', 'classifier__alpha': 0.01, 'classifier__hidden_layer_sizes': (512, 256, 128), 'classifier__

In [21]:
# Evaluate
best_model = grid_search.best_estimator_

y_pred_best = best_model.predict(X_test)
y_test_int  = y_test.astype(int)
y_pred_int  = y_pred_best.astype(int)

target_names = [str(cls) for cls in label_encoder.classes_]

print("Classification Report (best grid-search model):")
print(classification_report(y_test_int, y_pred_int, target_names=target_names))
print(f"Overall Accuracy : {accuracy_score(y_test_int, y_pred_int):.4f}")
print(f"F1 (delayed=1)   : {f1_score(y_test_int, y_pred_int, pos_label=1):.4f}")


Classification Report (best grid-search model):
              precision    recall  f1-score   support

         0.0       0.88      0.71      0.79   1012213
         1.0       0.37      0.63      0.47    269104

    accuracy                           0.70   1281317
   macro avg       0.62      0.67      0.63   1281317
weighted avg       0.77      0.70      0.72   1281317

Overall Accuracy : 0.6953
F1 (delayed=1)   : 0.4655


In [23]:
#Extra training as I only ran the grid search on ~50 epochs, probably enough with a simple model like this but want to try it anyways


import copy


#pull best model
best_ffn = best_model.named_steps['classifier']

#warm start
best_ffn.set_params(
    warm_start=True,
    max_iter=best_ffn.n_iter_ + 250,
    n_iter_no_change=20,
)


#Fit with warm start
best_model.fit(X_train_bal, y_train_bal,
               classifier__sample_weight=sample_weights)

from sklearn.metrics import classification_report, accuracy_score, f1_score

#New Eval
y_pred_warm = best_model.predict(X_test)
print("\nreport:")
print(classification_report(y_test.astype(int), y_pred_warm.astype(int),
                             target_names=[str(c) for c in label_encoder.classes_]))
print(f"Acc : {accuracy_score(y_test, y_pred_warm):.4f}")
print(f"F1: {f1_score(y_test, y_pred_warm, pos_label=1):.4f}")




report:
              precision    recall  f1-score   support

         0.0       0.88      0.72      0.79   1012213
         1.0       0.37      0.62      0.47    269104

    accuracy                           0.70   1281317
   macro avg       0.63      0.67      0.63   1281317
weighted avg       0.77      0.70      0.72   1281317

Acc : 0.7013
F1: 0.4662


In [ ]:
label_encoder.classes_

array([0., 1.])

In [ ]:
import joblib, os

EXPORT_DIR = "model_export"
os.makedirs(EXPORT_DIR, exist_ok=True)

PIPELINE_PATH      = os.path.join(EXPORT_DIR, "flight_delay_pipeline.joblib")
LABEL_ENCODER_PATH = os.path.join(EXPORT_DIR, "label_encoder.joblib")
FEATURE_META_PATH  = os.path.join(EXPORT_DIR, "feature_metadata.joblib")

joblib.dump(best_model,     PIPELINE_PATH,      compress=3)
joblib.dump(label_encoder,  LABEL_ENCODER_PATH, compress=3)
joblib.dump({
    "continuous": CONTINUOUS_FEATURES,
    "categorical": CATEGORICAL_FEATURES,
    "all": ALL_FEATURES,
    "target": TARGET,
    "delay_threshold_min": DELAY_THRESHOLD,
}, FEATURE_META_PATH)

print(f"Pipeline saved  →  {PIPELINE_PATH}")
print(f"LabelEncoder    →  {LABEL_ENCODER_PATH}")
print(f"Feature meta    →  {FEATURE_META_PATH}")

# Smoke-test: reload and run a single prediction
_pipeline_check = joblib.load(PIPELINE_PATH)
_enc_check      = joblib.load(LABEL_ENCODER_PATH)
_test_pred      = _pipeline_check.predict(X_test.iloc[:5])
print(f"\nSmoke-test predictions (first 5): {_enc_check.inverse_transform(_test_pred)}")

Classification Report:
              precision    recall  f1-score   support

         0.0       0.88      0.69      0.77   1012213
         1.0       0.36      0.65      0.46    269104

    accuracy                           0.68   1281317
   macro avg       0.62      0.67      0.62   1281317
weighted avg       0.77      0.68      0.71   1281317

Overall Accuracy: 0.6818


: 